# 📘 Tái hiện và Giải thích Chi tiết Pipeline XGB-DQN Cải tiến (WHU-LX)

Notebook này chứa toàn bộ pipeline tái hiện và giải thích chi tiết tác tử **DQN cải tiến** kết hợp **XGBoost Surrogate** để điều khiển hệ thống HVAC và cửa sổ tự động.

---

## Tổng quan kiến trúc hệ thống

```
┌──────────────────────────────────────────────────────────┐
│                    Pipeline XGB-DQN                      │
│                                                          │
│  CSV Data ──► XGBoost Surrogate ──► DQN Agent            │
│              (Giả lập nhiệt độ)     (Ra quyết định)      │
│                    ▲                      │              │
│                    │               Action (0-23)         │
│                    │                      │              │
│                    └──── next_T_in ◄──────┘              │
│                         (phản hồi)                       │
└──────────────────────────────────────────────────────────┘
```

**Hai thành phần chính:**
1. **XGBoost Surrogate** — học và dự đoán sự thay đổi nhiệt độ phòng (`ΔT_in`) theo điều kiện thời tiết và hành động điều khiển.
2. **DQN Agent** — học chính sách điều khiển tối ưu để chọn trong 24 hành động (bật/tắt AC ở các mức nhiệt độ khác nhau, đóng/mở cửa sổ).


---
## 1. Chuẩn bị thư viện và cấu trúc dữ liệu

### 1.1 Các thư viện được sử dụng

Pipeline sử dụng các nhóm thư viện chính:
- `pandas`, `numpy`: đọc và xử lý dữ liệu dạng bảng.
- `matplotlib`: trực quan hóa kết quả huấn luyện và đánh giá.
- `scikit-learn`: chia dữ liệu train/test, mã hóa nhãn và tính toán metric.
- `xgboost`: huấn luyện mô hình dự đoán sự thay đổi nhiệt độ phòng.
- `tensorflow`: xây dựng và huấn luyện mạng DQN.

Dữ liệu được đọc trực tiếp từ `data/Cleaned_data.csv` đã được đính kèm trong repo.

In [ ]:
import json
import math
import random
from collections import deque
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import tensorflow as tf
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUT_DIR = PROJECT_ROOT / "artifacts" / "outputs" / "whulx_reproduction"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / "Cleaned_data.csv"

np.random.seed(2022)
random.seed(2022)
tf.random.set_seed(2022)

print("Project root:", PROJECT_ROOT)
print("Dữ liệu:", DATA_PATH)
print("Thư mục lưu kết quả:", OUT_DIR)
print(f"TensorFlow version: {tf.__version__}")
print(f"XGBoost version:    {xgb.__version__}")

### 1.2 Vector State — Đầu vào của tác tử DQN

Tại mỗi bước thời gian `t`, tác tử nhận được vector trạng thái **8 chiều**:

| Index | Tên biến | Đơn vị | Mô tả |
|-------|----------|--------|-------|
| 0 | `Indoor_Temp` | °C | Nhiệt độ trong phòng |
| 1 | `Indoor_RH` | % | Độ ẩm tương đối trong phòng |
| 2 | `Outdoor_Temp` | °C | Nhiệt độ ngoài trời |
| 3 | `Outdoor_RH` | % | Độ ẩm tương đối ngoài trời |
| 4 | `Rain` | - | Lượng mưa |
| 5 | `Cloud` | - | Độ che phủ mây |
| 6 | `Windspeed` | m/s | Tốc độ gió |
| 7 | `Hour` | 0-23 | Giờ trong ngày |


---
## 2. Chuẩn Tiện nghi Nhiệt ASHRAE 55 — `comfort_bounds()`

### 2.1 Lý thuyết Adaptive Comfort Model

ASHRAE Standard 55 định nghĩa **dải nhiệt độ thoải mái thích ứng** phụ thuộc vào nhiệt độ trung bình ngoài trời (`T_out`). Khi trời nóng hơn, con người quen với nhiệt độ cao hơn, nên dải tiện nghi dịch lên trên.

```
T_out ≤ 10°C → [17.4, 18.4] ... [23.4, 24.4]  (trời mát, dải comfort thấp)
T_out ≥ 30°C → [23.6, 24.6] ... [29.6, 30.6]  (trời nóng, dải comfort cao)
10°C < T_out < 30°C → nội suy tuyến tính
```

Hàm trả về 4 giá trị: `(outer_lower, lower, upper, outer_upper)` — trong đó `lower` và `upper` là dải 80% tiện nghi.

In [ ]:
STANDARD_BANDS = {
    "ASHRAE": [[17.4, 18.4, 23.4, 24.4], [23.6, 24.6, 29.6, 30.6], [10, 30]]
}


def comfort_bounds(outdoor_temp, standard="ASHRAE"):
    l11, l12, u12, u11 = STANDARD_BANDS[standard][0]
    l21, l22, u22, u21 = STANDARD_BANDS[standard][1]
    t1, t2 = STANDARD_BANDS[standard][2]

    if outdoor_temp <= t1:
        return l11, l12, u12, u11
    if outdoor_temp >= t2:
        return l21, l22, u22, u21

    increase_l = (outdoor_temp - t1) * (l21 - l11) / (t2 - t1)
    increase_u = (outdoor_temp - t1) * (u21 - u11) / (t2 - t1)
    return l11 + increase_l, l12 + increase_l, u12 + increase_u, u11 + increase_u


def comfort_ok(indoor_temp, outdoor_temp):
    _, lower, upper, _ = comfort_bounds(float(outdoor_temp))
    return lower <= float(indoor_temp) <= upper


# === TRỰC QUAN HÓA DẢI TIỆN NGHI THEO NHIỆT ĐỘ NGOÀI TRỜI ===
t_out_range = np.linspace(5, 38, 200)
lowers, uppers = [], []
for t in t_out_range:
    _, l, u, _ = comfort_bounds(t)
    lowers.append(l)
    uppers.append(u)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.fill_between(t_out_range, lowers, uppers, alpha=0.3, color='green', label='Dải tiện nghi 80% ASHRAE 55')
ax.plot(t_out_range, lowers, 'g--', linewidth=1.5, label='Giới hạn dưới (L_c)')
ax.plot(t_out_range, uppers, 'g-',  linewidth=1.5, label='Giới hạn trên (U_c)')
ax.axvline(10, color='blue', linestyle=':', alpha=0.7, label='Ngưỡng nội suy dưới (10°C)')
ax.axvline(30, color='red',  linestyle=':', alpha=0.7, label='Ngưỡng nội suy trên (30°C)')
ax.set_xlabel('Nhiệt độ ngoài trời T_out (°C)', fontsize=12)
ax.set_ylabel('Nhiệt độ trong phòng (°C)', fontsize=12)
ax.set_title('Dải tiện nghi nhiệt thích ứng ASHRAE Standard 55', fontsize=14, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Không gian Hành động — Chi tiết 24 Actions

Tác tử có **24 hành động rời rạc** phối hợp điều khiển cả AC và cửa sổ được liệt kê chi tiết dưới đây:

| Nhóm hành động | Action ID | Điều hòa (AC) | Cửa sổ | Setpoint Nhiệt độ |
| :--- | :---: | :---: | :---: | :---: |
| **Nhóm 1: Tắt AC + Đóng cửa** | `0` | Tắt | Đóng | — |
| **Nhóm 2: Bật AC + Đóng cửa** | `1` | Bật | Đóng | 20 °C |
| | `2` | Bật | Đóng | 21 °C |
| | `3` | Bật | Đóng | 22 °C |
| | `4` | Bật | Đóng | 23 °C |
| | `5` | Bật | Đóng | 24 °C |
| | `6` | Bật | Đóng | 25 °C |
| | `7` | Bật | Đóng | 26 °C |
| | `8` | Bật | Đóng | 27 °C |
| | `9` | Bật | Đóng | 28 °C |
| | `10` | Bật | Đóng | 29 °C |
| | `11` | Bật | Đóng | 30 °C |
| **Nhóm 3: Tắt AC + Mở cửa** | `12` | Tắt | Mở | — |
| **Nhóm 4: Bật AC + Mở cửa** | `13` | Bật | Mở | 20 °C |
| | `14` | Bật | Mở | 21 °C |
| | `15` | Bật | Mở | 22 °C |
| | `16` | Bật | Mở | 23 °C |
| | `17` | Bật | Mở | 24 °C |
| | `18` | Bật | Mở | 25 °C |
| | `19` | Bật | Mở | 26 °C |
| | `20` | Bật | Mở | 27 °C |
| | `21` | Bật | Mở | 28 °C |
| | `22` | Bật | Mở | 29 °C |
| | `23` | Bật | Mở | 30 °C |


In [ ]:
def map_action_to_dataframe(action):
    action = int(action)
    target_temp, ac_status, window_status, c_last_time, w_last_time = 0, 0, 0, 0, 0

    if action == 0:
        pass
    elif 0 < action < 12:
        target_temp = 19 + action
        ac_status = 1
        c_last_time = 60
    elif action == 12:
        window_status = 1
        w_last_time = 60
    else:
        target_temp = action + 7
        ac_status = 1
        window_status = 1
        c_last_time = 60
        w_last_time = 60

    return target_temp, ac_status, window_status, c_last_time, w_last_time

---
## 4. Hàm Phần thưởng (Reward) — `calculate_reward()`

### 4.1 Cải tiến quan trọng nhất: Reward Rescaling

**Vấn đề của repo gốc WHU-LX:**
- Phạt lệch tiện nghi: `comfort_weight=1.0` → phạt chỉ `-2.25` khi lệch 1.5°C
- Phạt năng lượng AC: cố định `-52.2` mỗi bước
- Kết quả: DQN học được "không bao giờ bật AC" vì phạt AC > phạt lệch tiện nghi!

**Cải tiến của nhóm: `comfort_weight=120.0`**
- Phạt lệch tiện nghi 1.5°C: `-(1.5)² × 120 = -270` → **lớn hơn phạt AC (-52.2)**
- DQN có động lực bật AC khi trời nóng, tắt AC khi phòng đã đủ mát

```
R(s,a,s') = comfort_weight × P_comfort(s') + P_energy(a)
```

In [ ]:
def calculate_reward(state, action, next_state, comfort_weight=120.0):
    indoor_temp = float(next_state[0])
    outdoor_temp = float(next_state[2])
    _, lower, upper, _ = comfort_bounds(outdoor_temp)

    if lower <= indoor_temp <= upper:
        comfort_penalty = 0.0
    elif indoor_temp < lower:
        comfort_penalty = -((indoor_temp - lower) ** 2)
    else:
        comfort_penalty = -((indoor_temp - upper) ** 2)

    if action in {0, 12}:
        energy_penalty = 0.0
    elif action > 12:
        energy_penalty = -2 * (60 * 0.87 * 1)
    else:
        energy_penalty = -(60 * 0.87 * 1)

    reward = comfort_weight * comfort_penalty + energy_penalty
    return float(reward)

---
## 5. Chuẩn hóa State (State Normalization) — `normalize_state()`

### Tại sao cần chuẩn hóa?

Các biến trong state vector có **scale rất khác nhau**:
- `Indoor_RH` ≈ 50-90 (thang đo lớn)
- `Rain` ≈ 0-1.0 (thang đo nhỏ)
- `Hour` ≈ 0-23

Khi feed vào mạng MLP mà không chuẩn hóa, gradient bị chi phối bởi các biến có độ lớn cao, dẫn đến bão hòa gradient và làm chậm hoặc ngăn cản học. **Giải pháp: Min-Max scaling đưa tất cả về [0, 1].**

In [ ]:
def normalize_state(state):
    norm = np.copy(state).astype(np.float32)
    norm[0] = float(state[0]) / 40.0       # Indoor_Temp
    norm[1] = float(state[1]) / 100.0      # Indoor_RH
    norm[2] = float(state[2]) / 40.0       # Outdoor_Temp
    norm[3] = float(state[3]) / 100.0      # Outdoor_RH
    norm[4] = float(state[4]) / 10.0       # Rain
    norm[5] = float(state[5]) / 10.0       # Cloud
    norm[6] = float(state[6]) / 10.0       # Windspeed
    norm[7] = float(state[7]) / 23.0       # Hour
    return norm

---
## 6. Đọc và tiền xử lý dữ liệu

Dữ liệu đầu vào là `Cleaned_data.csv` từ WHU-LX. Dù tên file là cleaned, pipeline vẫn làm thêm một số bước để đảm bảo mô hình học được:

- Đọc CSV với encoding `gbk` vì dữ liệu gốc dùng encoding này.
- Chuyển `Date_Time` sang kiểu thời gian.
- Loại bỏ missing value và các giá trị lỗi `-999`.
- Mã hóa các cột dạng chuỗi sang số bằng `LabelEncoder`.

Kết quả gồm `raw` là dữ liệu gốc và `data` là dữ liệu đã xử lý để huấn luyện.

In [ ]:
def load_and_prepare_data(data_path):
    raw = pd.read_csv(data_path, encoding="gbk")
    data = raw.copy()

    data["Date_Time"] = pd.to_datetime(data["Date_Time"])
    data = data.dropna()
    data = data[data != -999].dropna()

    for col in data.columns:
        if data[col].dtype == "object":
            encoder = LabelEncoder()
            data[col] = encoder.fit_transform(data[col])

    return raw, data


raw, data = load_and_prepare_data(DATA_PATH)

print("Kích thước dữ liệu gốc:", raw.shape)
print("Kích thước sau xử lý:", data.shape)
data.head()

---
## 7. Huấn luyện XGBoost để dự đoán chuyển trạng thái

Trong pipeline này, XGBoost không phải là mô hình điều khiển. Nó đóng vai trò như một **mô hình môi trường gần đúng (Surrogate Environment)**.

- Input của XGBoost là trạng thái hiện tại cộng với thông tin action/trạng thái điều khiển.
- Target là `Differ_Indoor_Temp`, tức độ thay đổi nhiệt độ trong nhà ở bước tiếp theo (`ΔT_in`).
- Khi DQN chọn action, XGBoost dự đoán `Differ_Indoor_Temp` để cập nhật `Indoor_Temp` tiếp theo.

In [ ]:
def train_xgboost(data, device="cpu"):
    x_data = data.drop(
        ["Next_Indoor_Temp", "Next_Indoor_RH", "Date_Time", "Study_ID", "Differ_Indoor_Temp", "ID"],
        axis=1,
    )
    y_data = data["Differ_Indoor_Temp"]

    x_train, x_test, y_train, y_test = train_test_split(
        x_data, y_data, test_size=0.2, random_state=2022
    )

    model = xgb.XGBRegressor(
        random_state=2000,
        verbosity=0,
        n_jobs=-1,
        tree_method="hist",
        device=device,
        max_depth=5,
        learning_rate=0.23474,
        n_estimators=500,
    )
    model.fit(x_train, y_train)

    pred = model.predict(x_test)
    mse = mean_squared_error(y_test, pred)
    metrics = {
        "x_shape": list(x_data.shape),
        "train_shape": list(x_train.shape),
        "test_shape": list(x_test.shape),
        "mae": float(mean_absolute_error(y_test, pred)),
        "rmse": float(math.sqrt(mse)),
        "r2": float(r2_score(y_test, pred)),
    }
    return model, metrics


model_xgb, xgb_metrics = train_xgboost(data, device="cpu")
xgb_metrics

---
## 8. Tách dữ liệu thành các cửa sổ ngày

Để DQN tương tác với dữ liệu, ta cắt chuỗi thời gian thành các cửa sổ theo ngày (episode). Mỗi ngày gồm 24 bước thời gian, trong đó khung điều khiển thực tế diễn ra từ 6h đến 17h (12 timestep).

Biến `max_complete_days` cho biết số lượng ngày hoàn chỉnh có trong dữ liệu.

In [ ]:
def choose_day(start_index, data):
    data_test_00 = data.iloc[start_index : start_index + 23].copy()
    data_test_0a = data_test_00.reset_index(drop=True)
    data_test_0 = data_test_00.reset_index(drop=True)

    data_test_0.at[0, "CLast_Time_T"] = (
        data_test_0a.iloc[0]["CLast_Time_T"] - data_test_0a.iloc[0]["AC_Status"] * 60
    )
    data_test_0.at[0, "WLast_Time_T"] = (
        data_test_0a.iloc[0]["WLast_Time_T"] - data_test_0a.iloc[0]["Window_Status"] * 60
    )

    data_test = data_test_0[
        [
            "Indoor_Temp",
            "Indoor_RH",
            "Outdoor_Temp",
            "Outdoor_RH",
            "Rain",
            "Cloud",
            "Windspeed",
            "Hour",
            "Next_Outdoor_Temp",
            "Next_Outdoor_RH",
        ]
    ].copy()

    xgboost_test = data_test_0.drop(
        ["Next_Indoor_Temp", "Next_Indoor_RH", "Date_Time", "Study_ID", "Differ_Indoor_Temp", "ID"],
        axis=1,
    )
    return data_test, xgboost_test


max_complete_days = max(1, (len(data) - 24) // 24)
print(f"Số ngày hoàn chỉnh có thể rollout: {max_complete_days}")

---
## 9. Xây dựng Replay Buffer và mạng DQN

### 9.1 Vai trò của Replay Buffer
Replay Buffer lưu trữ các kinh nghiệm `(s, a, s', r)`. Khi huấn luyện, việc lấy mẫu ngẫu nhiên (random sampling) giúp:
1. **Phá vỡ tính tương quan thời gian** — tránh overfitting vào pattern gần nhất.
2. **Tái sử dụng dữ liệu** — mỗi experience có thể được học nhiều lần.
3. **Ổn định hóa gradient**.

### 9.2 Kiến trúc mạng DQN
Kiến trúc mạng là MLP 2 lớp ẩn (mỗi lớp 64 units, kích hoạt ReLU). Đầu ra là 24 giá trị Q tương ứng với 24 hành động rời rạc.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, next_state, reward):
        self.buffer.append((state, action, next_state, reward))

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)


class DQN(tf.keras.Model):
    def __init__(self, num_actions):
        super().__init__()
        self.dense1 = tf.keras.layers.Dense(64, activation="relu")
        self.dense2 = tf.keras.layers.Dense(64, activation="relu")
        self.output_layer = tf.keras.layers.Dense(num_actions, activation="linear")

    def call(self, inputs):
        x = self.dense1(inputs)
        x = self.dense2(x)
        return self.output_layer(x)

---
## 10. Hàm cập nhật Q-network và chính sách epsilon-greedy

### 10.1 Cập nhật Q-Network bằng Target Network tách biệt
**Cải tiến so với WHU-LX gốc:**
- Sử dụng mạng **Target Network** tách biệt có trọng số cập nhật chậm (đồng bộ mỗi 10 episodes) để tính toán TD target, tránh hiện tượng "Moving Target Problem".

### 10.2 Epsilon-Greedy Policy
Cân bằng giữa Exploration (khám phá ngẫu nhiên) và Exploitation (khai thác chính sách tối ưu). Epsilon giảm dần từ 1.0 xuống 0.1 qua quá trình train.

In [ ]:
def update_q_network(q_network, target_q_network, replay_buffer, optimizer, loss_fn, gamma, num_actions, batch_size):
    states, actions, next_states, rewards = zip(*replay_buffer.sample(batch_size))

    norm_states = np.array([normalize_state(s) for s in states])
    norm_next_states = np.array([normalize_state(s) for s in next_states])

    states_tensor = tf.convert_to_tensor(norm_states, dtype=tf.float32)
    next_states_tensor = tf.convert_to_tensor(norm_next_states, dtype=tf.float32)
    actions = tf.convert_to_tensor(np.array(actions), dtype=tf.int32)
    rewards = tf.convert_to_tensor(np.array(rewards), dtype=tf.float32)

    with tf.GradientTape() as tape:
        q_values = q_network(states_tensor)
        target_q_values = target_q_network(next_states_tensor)
        target_q_values = rewards + gamma * tf.reduce_max(target_q_values, axis=1)
        mask = tf.one_hot(actions, num_actions)
        q_action = tf.reduce_sum(q_values * mask, axis=1)
        loss = loss_fn(target_q_values, q_action)

    grads = tape.gradient(loss, q_network.trainable_variables)
    optimizer.apply_gradients(zip(grads, q_network.trainable_variables))
    return float(loss.numpy())


def epsilon_greedy_policy(q_network, state, epsilon, num_actions):
    if np.random.rand() < epsilon:
        return int(np.random.randint(num_actions))
    norm_state = normalize_state(state)
    q_values = q_network(np.array([norm_state], dtype=np.float32))
    return int(np.argmax(q_values[0]))

---
## 11. Rollout một cửa sổ ngày bằng XGBoost + DQN

`rollout_day()` là trung tâm mô phỏng:
1. **Warm-up (0-5h)**: Cập nhật bằng XGBoost tự nhiên (AC tắt, cửa đóng).
2. **Khung điều khiển (6-17h)**: Tác tử chọn hành động (AC Setpoint + Cửa đóng/mở), mô hình XGBoost dự đoán sự thay đổi nhiệt độ phòng kế tiếp.

In [ ]:
def rollout_day(
    model_xgb,
    q_network,
    target_q_network,
    data_test,
    xgboost_test,
    epsilon,
    train=False,
    replay_buffer=None,
    train_cfg=None,
    fixed_action=None,
):
    data_pre_test = data_test.copy()
    xgboost_pre_test = xgboost_test.copy()
    num_features = 8
    actions = []
    rewards = []
    losses = []

    for step in range(6):
        xgboost_pre_test.loc[
            step,
            ["Target_Temp", "AC_Status", "Window_Status", "CLast_Time", "WLast_Time", "CLast_Time_T", "WLast_Time_T"],
        ] = 0
        hour_row_df = pd.DataFrame(xgboost_pre_test.iloc[step]).T
        next_differ_temp = model_xgb.predict(hour_row_df)[0]
        next_in_temp = xgboost_pre_test.iloc[step]["Indoor_Temp"] + next_differ_temp
        xgboost_pre_test.at[step + 1, "Indoor_Temp"] = next_in_temp
        data_pre_test.at[step + 1, "Indoor_Temp"] = next_in_temp

    state = data_pre_test.iloc[6, :num_features].values

    for step in range(6, 18):
        if fixed_action is None:
            action = epsilon_greedy_policy(q_network, state, epsilon, train_cfg["num_actions"])
        else:
            action = int(fixed_action)

        target_temp, ac_status, window_status, c_last_time, w_last_time = map_action_to_dataframe(action)
        actions.append(action)

        xgboost_pre_test.at[step, "Target_Temp"] = target_temp
        xgboost_pre_test.at[step, "AC_Status"] = ac_status
        xgboost_pre_test.at[step, "Window_Status"] = window_status
        xgboost_pre_test.at[step, "CLast_Time"] = c_last_time
        xgboost_pre_test.at[step, "WLast_Time"] = w_last_time
        xgboost_pre_test.at[step, "CLast_Time_T"] = (
            xgboost_pre_test.iloc[step - 1]["CLast_Time_T"] + c_last_time if c_last_time > 0 else 0
        )
        xgboost_pre_test.at[step, "WLast_Time_T"] = (
            xgboost_pre_test.iloc[step - 1]["WLast_Time_T"] + w_last_time if w_last_time > 0 else 0
        )

        hour_row_df = pd.DataFrame(xgboost_pre_test.iloc[step]).T
        next_differ_temp = model_xgb.predict(hour_row_df)[0]
        next_in_temp = xgboost_pre_test.iloc[step]["Indoor_Temp"] + next_differ_temp
        xgboost_pre_test.at[step + 1, "Indoor_Temp"] = next_in_temp
        data_pre_test.at[step + 1, "Indoor_Temp"] = next_in_temp

        next_state = data_pre_test.iloc[step + 1, :num_features].values
        reward = calculate_reward(state, action, next_state)
        rewards.append(reward)

        if train and replay_buffer is not None:
            replay_buffer.push(state, action, next_state, reward)
            if len(replay_buffer) >= train_cfg["batch_size"]:
                losses.append(
                    update_q_network(
                        q_network,
                        target_q_network,
                        replay_buffer,
                        train_cfg["optimizer"],
                        train_cfg["loss_fn"],
                        train_cfg["gamma"],
                        train_cfg["num_actions"],
                        train_cfg["batch_size"],
                    )
                )
        state = next_state

    return data_pre_test, actions, rewards, losses

---
## 12. Hàm tổng hợp metric

Các hàm hỗ trợ tính toán chỉ số tiện nghi (comfort %), tỉ lệ bật AC (ac_on %), và tỉ lệ mở cửa sổ (window_open %) để đánh giá so sánh.

In [ ]:
def summarize_day(data_day, actions):
    control = data_day.iloc[6:18].copy()
    comfort = [comfort_ok(row["Indoor_Temp"], row["Outdoor_Temp"]) for _, row in control.iterrows()]
    ac_on = sum(1 for a in actions if 0 < a < 12 or a > 12)
    window_open = sum(1 for a in actions if a >= 12)

    return {
        "comfort_pct": float(np.mean(comfort) * 100),
        "ac_on_pct": float(ac_on / len(actions) * 100) if actions else 0.0,
        "window_open_pct": float(window_open / len(actions) * 100) if actions else 0.0,
        "mean_indoor_temp": float(control["Indoor_Temp"].mean()),
    }


def summarize_human(data_test):
    control = data_test.iloc[6:18].copy()
    comfort = [comfort_ok(row["Indoor_Temp"], row["Outdoor_Temp"]) for _, row in control.iterrows()]
    return {
        "comfort_pct": float(np.mean(comfort) * 100),
        "mean_indoor_temp": float(control["Indoor_Temp"].mean()),
    }


def summarize_human_with_actions(data_test, xgboost_test):
    metrics = summarize_human(data_test)
    control = xgboost_test.iloc[6:18].copy()
    metrics["ac_on_pct"] = float(control["AC_Status"].mean() * 100)
    metrics["window_open_pct"] = float(control["Window_Status"].mean() * 100)
    return metrics


def aggregate_metric_rows(rows):
    if not rows:
        return {}
    keys = sorted({key for row in rows for key in row if isinstance(row.get(key), (int, float, np.number))})
    return {key: float(np.mean([row[key] for row in rows if key in row])) for key in keys}

---
## 13. Huấn luyện DQN trên toàn bộ tập ngày

### Cải tiến quan trọng: Multi-day Training
**Vấn đề của WHU-LX gốc:** Cố định `day_index=0` → Overfit vào 1 ngày mát trời (dẫn đến chính sách 0% bật AC).
**Cải tiến:** Mỗi episode **chọn ngẫu nhiên** một ngày từ toàn bộ `max_complete_days` ngày trong dữ liệu.

In [ ]:
num_actions = 24
EPISODES = max_complete_days  # Có thể tăng/giảm tùy ý

q_network = DQN(num_actions)
q_network(np.zeros((1, 8), dtype=np.float32))

target_q_network = DQN(num_actions)
target_q_network(np.zeros((1, 8), dtype=np.float32))
target_q_network.set_weights(q_network.get_weights())

replay_buffer = ReplayBuffer(10000)
optimizer = tf.optimizers.Adam(0.001)
loss_fn = tf.losses.MeanSquaredError()

epsilon = 1.0
min_epsilon = 0.1
epsilon_decay = 0.995

train_cfg = {
    "num_actions": num_actions,
    "batch_size": 32,
    "gamma": 0.9,
    "optimizer": optimizer,
    "loss_fn": loss_fn,
}

history = []

for episode in range(EPISODES):
    train_day_index = random.randint(0, max_complete_days - 1)
    train_data_test, train_xgboost_test = choose_day(train_day_index * 24, data)

    _, actions, rewards, losses = rollout_day(
        model_xgb,
        q_network,
        target_q_network,
        train_data_test,
        train_xgboost_test,
        epsilon,
        train=True,
        replay_buffer=replay_buffer,
        train_cfg=train_cfg,
    )

    if epsilon > min_epsilon:
        epsilon *= epsilon_decay

    # Đồng bộ target network định kỳ mỗi 10 episodes
    if (episode + 1) % 10 == 0:
        target_q_network.set_weights(q_network.get_weights())

    history.append(
        {
            "episode": episode + 1,
            "day_index": train_day_index,
            "reward": float(np.sum(rewards)),
            "epsilon": float(epsilon),
            "avg_loss": float(np.mean(losses)) if losses else np.nan,
        }
    )

    if (episode + 1) % 100 == 0 or episode + 1 == EPISODES:
        print(
            f"Episode {episode + 1}/{EPISODES}, "
            f"day={train_day_index}, reward={np.sum(rewards):.2f}, epsilon={epsilon:.3f}"
        )

history_df = pd.DataFrame(history)
history_df.tail()

---
## 14. Đánh giá toàn bộ pipeline trên tất cả ngày hoàn chỉnh

Sau khi tác tử đã được huấn luyện, tiến hành rollout đánh giá trên toàn bộ tập ngày hoàn chỉnh (`max_complete_days`) và so sánh với các baseline.

In [ ]:
def evaluate_many_days(model_xgb, q_network, target_q_network, data, day_indices, num_actions):
    controllers = {
        "DQN": None,
        "Human": "human",
        "Off_Closed": 0,
        "Window_Open": 12,
        "AC_25_Closed": 6,
        "AC_26_Closed": 7,
        "AC_27_Closed": 8,
    }
    rows_by_controller = {name: [] for name in controllers}
    actions_by_controller = {name: [] for name in controllers}

    for day_index in day_indices:
        data_test_i, xgboost_test_i = choose_day(day_index * 24, data)

        for name, fixed_action in controllers.items():
            if fixed_action == "human":
                rows_by_controller[name].append(summarize_human_with_actions(data_test_i, xgboost_test_i))
                continue

            eval_day_i, actions_i, rewards_i, _ = rollout_day(
                model_xgb,
                q_network,
                target_q_network,
                data_test_i,
                xgboost_test_i,
                epsilon=0.0,
                train=False,
                train_cfg={"num_actions": num_actions},
                fixed_action=fixed_action,
            )
            metrics = summarize_day(eval_day_i, actions_i)
            metrics["total_reward"] = float(np.sum(rewards_i))
            rows_by_controller[name].append(metrics)
            actions_by_controller[name].extend(int(action) for action in actions_i)

    summary = []
    action_distributions = {}
    for name, rows in rows_by_controller.items():
        metrics = aggregate_metric_rows(rows)
        summary.append({"controller": name, **metrics})

        actions = actions_by_controller.get(name, [])
        if actions:
            counts = pd.Series(actions).value_counts().sort_index()
            action_distributions[name] = {
                int(action): float(count / len(actions) * 100)
                for action, count in counts.items()
            }

    return summary, action_distributions


eval_indices = list(range(max_complete_days))
evaluation_summary, evaluation_action_distribution = evaluate_many_days(
    model_xgb,
    q_network,
    target_q_network,
    data,
    eval_indices,
    num_actions,
)

evaluation_df = pd.DataFrame(evaluation_summary)
evaluation_df

---
## 15. Trực quan hóa kết quả bằng đồ thị

Sau khi có `history_df`, `evaluation_df` và `evaluation_action_distribution`, ta biểu diễn kết quả dưới dạng đồ thị để đánh giá trực quan.

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")


def save_current_figure(filename):
    path = OUT_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    print("Đã lưu:", path)


plot_files = []

### 15.1 Đồ thị quá trình huấn luyện (Learning Curves)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

history_plot = history_df.copy()
history_plot["reward_smooth"] = history_plot["reward"].rolling(window=50, min_periods=1).mean()

axes[0].plot(history_plot["episode"], history_plot["reward"], alpha=0.25, label="Reward từng episode")
axes[0].plot(history_plot["episode"], history_plot["reward_smooth"], linewidth=2, label="Reward rolling mean")
axes[0].set_ylabel("Reward")
axes[0].set_title("Quá trình huấn luyện DQN")
axes[0].legend()

axes[1].plot(history_plot["episode"], history_plot["epsilon"], color="tab:orange")
axes[1].set_ylabel("Epsilon")

axes[2].plot(history_plot["episode"], history_plot["avg_loss"], color="tab:green")
axes[2].set_ylabel("Avg loss")
axes[2].set_xlabel("Episode")

save_current_figure("training_history_plot.png")
plot_files.append("training_history_plot.png")
plt.show()

### 15.2 So sánh comfort, AC và cửa sổ giữa các controller

In [ ]:
metric_cols = ["comfort_pct", "ac_on_pct", "window_open_pct"]
comparison_df = evaluation_df.set_index("controller")[metric_cols]

ax = comparison_df.plot(kind="bar", figsize=(11, 5), width=0.78)
ax.set_title("So sánh comfort, tỉ lệ bật AC và tỉ lệ mở cửa sổ")
ax.set_ylabel("Tỉ lệ (%)")
ax.set_xlabel("Controller")
ax.legend(["Comfort", "AC on", "Window open"], loc="upper right")
plt.xticks(rotation=35, ha="right")

save_current_figure("controller_comparison_metrics.png")
plot_files.append("controller_comparison_metrics.png")
plt.show()

### 15.3 So sánh total reward

In [ ]:
reward_df = evaluation_df.dropna(subset=["total_reward"]).set_index("controller")["total_reward"].sort_values()

ax = reward_df.plot(kind="barh", figsize=(10, 5), color="tab:purple")
ax.set_title("Total reward theo controller")
ax.set_xlabel("Total reward trung bình")
ax.set_ylabel("Controller")

save_current_figure("controller_total_reward.png")
plot_files.append("controller_total_reward.png")
plt.show()

### 15.4 Phân phối action của DQN

In [ ]:
dqn_action_dist = evaluation_action_distribution.get("DQN", {})

if dqn_action_dist:
    action_dist_df = (
        pd.Series(dqn_action_dist, name="percentage")
        .rename_axis("action")
        .reset_index()
        .sort_values("action")
    )

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.bar(action_dist_df["action"].astype(str), action_dist_df["percentage"], color="tab:blue")
    ax.set_title("Phân phối action của DQN trên toàn bộ ngày đánh giá")
    ax.set_xlabel("Action")
    ax.set_ylabel("Tỉ lệ chọn (%)")

    save_current_figure("dqn_action_distribution.png")
    plot_files.append("dqn_action_distribution.png")
    plt.show()
else:
    print("Chưa có action distribution cho DQN.")

---
## 16. Lưu kết quả full pipeline

Lưu các kết quả đánh giá để phục vụ báo cáo.

In [ ]:
result = {
    "episodes": EPISODES,
    "train_days": max_complete_days,
    "raw_shape": list(raw.shape),
    "after_clean_shape": list(data.shape),
    "xgboost": xgb_metrics,
    "reported_whulx_readme": {
        "comfort_duration_increase_pct": 24.0,
        "ac_usage_decrease_pct": 24.7,
    },
    "evaluation": {
        "eval_days": max_complete_days,
        "controllers": evaluation_summary,
        "action_distribution_pct": evaluation_action_distribution,
    },
}

history_df.to_csv(OUT_DIR / "training_history.csv", index=False)
evaluation_df.to_csv(OUT_DIR / "summary_metrics.csv", index=False)
(OUT_DIR / "result.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
q_network.save_weights(OUT_DIR / "whulx_dqn_reproduction.weights.h5")

print("Đã lưu kết quả full pipeline vào:", OUT_DIR)
print(json.dumps(result, indent=2, ensure_ascii=False))

---
## 17. So sánh với repo gốc WHU-LX

### 17.1 Bảng so sánh 4 cải tiến quan trọng

| Thành phần | WHU-LX Gốc (chứa lỗi) | Cải tiến (Chúng tôi) | Kết quả đạt được |
|---|---|---|---|
| **Training Strategy** | Cố định `day_index=0` (Overfit 1 ngày mát) | `random.randint(0, max_days)` | Tổng quát hóa toàn bộ thời tiết |
| **Reward Function** | `comfort_weight=1.0` (Phạt AC >> Phạt comfort) | `comfort_weight=120.0` | Cân bằng tiện nghi vs Năng lượng (DQN biết bật AC khi cần) |
| **Target Network** | Không có (Target di động gây mất ổn định) | `target_q_network` đồng bộ định kỳ | Quá trình học hội tụ ổn định |
| **State Normalization** | Không có (Scale đầu vào lệch cực lớn) | `normalize_state()` về `[0, 1]` | Dòng gradient tốt hơn, học nhanh hơn |

### 17.2 Giải thích kết quả
Với các cải tiến trên, tác tử DQN không còn hành vi tắt điều hòa 100% để tối ưu hóa năng lượng cực đoan nữa. Nó đã học được cách cân bằng giữa nhiệt độ dễ chịu trong phòng (comfort) và điện năng tiêu thụ.